In [ ]:
'''
def setup_wandb(project_name="parking_gaurd", run_name=None):
    """
    Weights & Biases 초기화
    
    Args:
        project_name (str): wandb 프로젝트 이름
        run_name (str): 실행 이름 (None이면 자동 생성)
    """
    try:
        wandb.init(
            project=project_name,
            name=run_name,
            config={
                "model": "YOLOv8-seg",
                "classes": ["solid_yellow_lane", "dotted_yellow_lane", "double_yellow_lane", 
                           "crosswalk", "sidewalk", "firehydrant", "car", "license_plate"],
                "task": "instance_segmentation"
            }
        )
        print("✓ wandb 초기화 완료")
        return True
    except Exception as e:
        print(f"wandb 초기화 실패: {e}")
        print("wandb 없이 계속 진행합니다...")
        return False

def split_dataset(source_images_dir, changed_labels_dir, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    """
    train/val/test로 분할
    Args:
        source_images_dir (str): 원본 이미지 폴더 경로
        changed_labels_dir (str): 원본 라벨 폴더 경로  
        output_dir (str): 출력 폴더 경로
        train_ratio (float): 훈련 데이터 비율
        val_ratio (float): 검증 데이터 비율
        test_ratio (float): 테스트 데이터 비율
        seed (int): 랜덤 시드
    """
    
    print(f"\n=== 데이터셋 분할 시작 ===")
    print(f"분할 비율 - Train: {train_ratio}, Val: {val_ratio}, Test: {test_ratio}")
    
    # 시드 설정
    random.seed(seed)
    
    # 이미지 파일 목록 가져오기
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
    image_files = []
    
    for file in os.listdir(source_images_dir):
        if any(file.lower().endswith(ext) for ext in image_extensions):
            # 대응하는 라벨 파일이 존재하는지 확인
            label_file = os.path.splitext(file)[0] + '.txt'
            label_path = os.path.join(changed_labels_dir, label_file)
            if os.path.exists(label_path):
                image_files.append(file)
            else:
                print(f"경고: {file}에 대응하는 라벨 파일이 없습니다.")
    
    print(f"총 이미지-라벨 쌍: {len(image_files)}개")
    
    if len(image_files) == 0:
        raise ValueError("유효한 이미지-라벨 쌍이 없습니다!")
    
    # 데이터 분할
    train_files, temp_files = train_test_split(image_files, test_size=(1-train_ratio), random_state=seed)
    
    if test_ratio > 0:
        val_files, test_files = train_test_split(temp_files, test_size=test_ratio/(val_ratio+test_ratio), random_state=seed)
    else:
        val_files = temp_files
        test_files = []
    
    print(f"분할 결과:")
    print(f"  Train: {len(train_files)}개")
    print(f"  Val: {len(val_files)}개")
    print(f"  Test: {len(test_files)}개")
    
    # 출력 디렉토리 생성
    splits = {
        'train': train_files,
        'val': val_files,
        'test': test_files
    }
    
    for split_name, file_list in splits.items():
        if len(file_list) == 0:
            continue
            
        # 디렉토리 생성
        img_dir = os.path.join(output_dir, 'images', split_name)
        label_dir = os.path.join(output_dir, 'labels', split_name)
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(label_dir, exist_ok=True)
        
        # 파일 복사
        for file_name in file_list:
            # 이미지 복사
            src_img = os.path.join(source_images_dir, file_name)
            dst_img = os.path.join(img_dir, file_name)
            shutil.copy2(src_img, dst_img)
            
            # 라벨 복사
            label_name = os.path.splitext(file_name)[0] + '.txt'
            src_label = os.path.join(changed_labels_dir, label_name)
            dst_label = os.path.join(label_dir, label_name)
            shutil.copy2(src_label, dst_label)
    
    print("✓ 데이터셋 분할 완료")
    return len(train_files), len(val_files), len(test_files)

import json
import os
from pathlib import Path

def convert_json_to_yolo(json_labels_dir, output_labels_dir, image_width, image_height):
    """
    JSON 라벨을 YOLO 형식으로 변환
    
    Args:
        json_labels_dir (str): JSON 라벨 파일들이 있는 디렉토리
        output_labels_dir (str): YOLO 형식 txt 파일을 저장할 디렉토리
        image_width (int): 이미지 너비 (정규화용)
        image_height (int): 이미지 높이 (정규화용)
    """
    
    class_mapping = {
        "solid_yellow_lane": 0,
        "dotted_yellow_lane": 1, 
        "double_yellow_lane": 2,
        "crosswalk": 3,
        "sidewalk": 4,
        "firehydrant": 5,
        "car": 6,
        "license_plate": 7
    }
    
    os.makedirs(output_labels_dir, exist_ok=True)
    
    json_files = [f for f in os.listdir(json_labels_dir) if f.endswith('.json')]
    converted_count = 0
    
    for json_file in json_files:
        json_path = os.path.join(json_labels_dir, json_file)
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # txt 파일명 생성
            txt_filename = os.path.splitext(json_file)[0] + '.txt'
            txt_path = os.path.join(output_labels_dir, txt_filename)
            
            with open(txt_path, 'w') as f:
                # JSON 구조에 따라 수정 필요
                # 예시: labelme 형식인 경우
                if 'shapes' in data:
                    for shape in data['shapes']:
                        label = shape['label']
                        if label in class_mapping:
                            class_id = class_mapping[label]
                            
                            # 폴리곤 좌표 정규화
                            points = shape['points']
                            normalized_coords = []
                            
                            for point in points:
                                x_norm = point[0] / image_width
                                y_norm = point[1] / image_height
                                normalized_coords.extend([x_norm, y_norm])
                            
                            # YOLO 형식으로 저장
                            coords_str = ' '.join(map(str, normalized_coords))
                            f.write(f"{class_id} {coords_str}\n")
                
                # COCO 형식인 경우
                elif 'annotations' in data:
                    for annotation in data['annotations']:
                        if 'segmentation' in annotation:
                            category_id = annotation['category_id']
                            segmentation = annotation['segmentation'][0]  # 첫 번째 폴리곤
                            
                            # 좌표 정규화
                            normalized_coords = []
                            for i in range(0, len(segmentation), 2):
                                x_norm = segmentation[i] / image_width
                                y_norm = segmentation[i+1] / image_height
                                normalized_coords.extend([x_norm, y_norm])
                            
                            coords_str = ' '.join(map(str, normalized_coords))
                            f.write(f"{category_id} {coords_str}\n")
            
            converted_count += 1
            
        except Exception as e:
            print(f"변환 실패: {json_file} - {str(e)}")
    
    print(f"✓ JSON → YOLO 변환 완료: {converted_count}/{len(json_files)} 파일")
    return converted_count

def get_image_dimensions(image_path):
    """이미지 크기 가져오기"""
    from PIL import Image
    
    with Image.open(image_path) as img:
        return img.width, img.height

def convert_dataset_json_to_yolo(source_images_dir, source_labels_dir, output_labels_dir):
    """
    전체 데이터셋의 JSON을 YOLO 형식으로 변환
    """
    print("=== JSON → YOLO 형식 변환 ===")
    
    image_files = [f for f in os.listdir(source_images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    os.makedirs(output_labels_dir, exist_ok=True)
    converted_count = 0
    
    for image_file in image_files:
        # 대응하는 JSON 파일 찾기
        json_filename = os.path.splitext(image_file)[0] + '.json'
        json_path = os.path.join(source_labels_dir, json_filename)
        
        if not os.path.exists(json_path):
            print(f"경고: {image_file}에 대응하는 JSON 파일이 없습니다.")
            continue
        
        # 이미지 크기 가져오기
        image_path = os.path.join(source_images_dir, image_file)
        try:
            width, height = get_image_dimensions(image_path)
            
            # JSON → YOLO 변환
            txt_filename = os.path.splitext(image_file)[0] + '.txt'
            txt_path = os.path.join(output_labels_dir, txt_filename)
            
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # JSON 형식에 따른 변환 (여기서는 labelme 형식 예시)
            with open(txt_path, 'w') as f:
                if 'shapes' in data:
                    for shape in data['shapes']:
                        label = shape['label']
                        class_mapping = {
                            "solid_yellow_lane": 0,
                            "dotted_yellow_lane": 1, 
                            "double_yellow_lane": 2,
                            "crosswalk": 3,
                            "sidewalk": 4,
                            "firehydrant": 5,
                            "car": 6,
                            "license_plate": 7
                        }
                        
                        if label in class_mapping:
                            class_id = class_mapping[label]
                            points = shape['points']
                            
                            # 좌표 정규화
                            normalized_coords = []
                            for point in points:
                                x_norm = point[0] / width
                                y_norm = point[1] / height
                                normalized_coords.extend([x_norm, y_norm])
                            
                            coords_str = ' '.join(map(str, normalized_coords))
                            f.write(f"{class_id} {coords_str}\n")
            
            converted_count += 1
            
        except Exception as e:
            print(f"변환 실패: {image_file} - {str(e)}")
    
    print(f"✓ 총 {converted_count}개 파일 변환 완료")
    return converted_count

def validate_polygon_labels(labels_dir, class_count=8):
    """
    폴리곤 형태 라벨 파일 검증
    
    Args:
        labels_dir (str): 라벨 디렉토리 경로
        class_count (int): 예상 클래스 개수
    """
    
    print(f"\n=== 폴리곤 라벨 검증 ===")
    
    label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
    
    if len(label_files) == 0:
        print("라벨 파일이 없습니다!")
        return False
    
    valid_files = 0
    total_objects = 0
    class_distribution = {}
    
    
    for label_file in label_files[:10]:  # 처음 10개 파일만 검사
        label_path = os.path.join(labels_dir, label_file)
        
        try:
            with open(label_path, 'r') as f:
                lines = f.readlines()
                
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 7:  # 최소 class_id + 3개 점 (6개 좌표)
                    print(f"경고: {label_file}에 잘못된 라벨 형식이 있습니다.")
                    continue
                
                class_id = int(parts[0])
                if class_id >= class_count:
                    print(f"경고: {label_file}에 범위를 벗어난 클래스 ID({class_id})가 있습니다.")
                    continue
                
                # 폴리곤 좌표 개수 확인 (짝수여야 함)
                coords = parts[1:]
                if len(coords) % 2 != 0:
                    print(f"경고: {label_file}에 잘못된 좌표 개수가 있습니다.")
                    continue
                
                # 클래스 분포 계산
                class_distribution[class_id] = class_distribution.get(class_id, 0) + 1
                total_objects += 1
            
            valid_files += 1
            
        except Exception as e:
            print(f"오류: {label_file} 읽기 실패 - {e}")
    
    print(f"검증 완료: {valid_files}/{len(label_files)} 파일 유효")
    print(f"총 객체 수: {total_objects}")
    print("클래스 분포:")
    
    class_names = ["solid_yellow_lane", "dotted_yellow_lane", "double_yellow_lane", 
                   "crosswalk", "sidewalk", "firehydrant", "car", "license_plate"]
    
    for class_id in sorted(class_distribution.keys()):
        count = class_distribution[class_id]
        class_name = class_names[class_id] if class_id < len(class_names) else f"unknown_{class_id}"
        print(f"  {class_id} ({class_name}): {count}개")
    
    return valid_files > 0

def create_dataset_yaml(dataset_path, yaml_path):
    """
    YOLO 학습을 위한 dataset.yaml 파일 생성
    
    Args:
        dataset_path (str): 데이터셋 루트 경로
        yaml_path (str): yaml 파일이 저장될 경로
    """
    
    class_names = [
        'solid_yellow_lane',      # 노란 실선
        'dotted_yellow_lane',     # 노란 점선  
        'double_yellow_lane',     # 노란 이중선
        'crosswalk',              # 횡단보도
        'sidewalk',               # 인도
        'firehydrant',            # 소화전
        'car',                    # 자동차
        'license_plate'           # 차량 번호판
    ]
    
    dataset_config = {
        'path': dataset_path,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': len(class_names),
        'names': class_names
    }
    
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✓ Dataset YAML 파일 생성: {yaml_path}")
    print(f"클래스 개수: {len(class_names)}")

def setup_training_environment():
    """
    학습 환경 설정 및 확인
    """
    print("=== 학습 환경 설정 ===")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"사용 디바이스: {device}")
    
    if device == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    return device

def train_yolo_segmentation(
    dataset_yaml_path,
    model_name='yolov8n-seg.pt',
    epochs=100,
    batch_size=16,
    img_size=640,
    project='runs/segment',
    name='road_segmentation',
    use_wandb=True
):
    """
    YOLO8 Segmentation 모델 학습 (기본값과 다른 파라미터만 명시)
    """
    
    print(f"\n=== YOLO8 Segmentation 학습 시작 ===")
    print(f"모델: {model_name}")
    print(f"에포크: {epochs}, 배치: {batch_size}, 이미지 크기: {img_size}")
    
    # GPU 사용 설정
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"사용 디바이스: {device}")

    try:
        model = YOLO(model_name)
        print(f"✓ {model_name} 모델 로드 완료")
        
        # 학습 설정 (기본값과 다른 것만 명시)
        train_args = {
            'data': dataset_yaml_path,
            'epochs': epochs,
            'batch': batch_size,
            'imgsz': img_size,
            'project': project,
            'name': name,
            'device': device,
            'exist_ok': True,
            'seed': 42,
            'deterministic': True,
            'save_period': 10,  # 10 에포크마다 저장
        }
        
        if use_wandb:
            os.environ["WANDB_MODE"] = "online"
        else:
            os.environ["WANDB_MODE"] = "disabled"
        
        # 학습 실행
        results = model.train(**train_args)
        
        print("✓ 학습 완료!")
        
        best_model_path = results.save_dir / 'weights' / 'best.pt'
        last_model_path = results.save_dir / 'weights' / 'last.pt'
        
        print(f"최고 성능 모델: {best_model_path}")
        print(f"마지막 모델: {last_model_path}")
        
        return results, str(best_model_path)
        
    except Exception as e:
        print(f"학습 중 오류 발생: {str(e)}")
        return None, None

def evaluate_model(model_path, dataset_yaml_path):
    """
    학습된 모델 성능 평가
    
    Args:
        model_path (str): 평가할 모델 경로
        dataset_yaml_path (str): 데이터셋 YAML 파일 경로
    """
    
    print(f"\n=== 모델 성능 평가 ===")
    
    try:
        model = YOLO(model_path)
        
        device = 'cuda' if torch.cuda.is_available() else 'cpu'


        # 평가 실행 (기본값과 다른 설정만 명시)
        metrics = model.val(
            data=dataset_yaml_path,
            device=device,
            save_json=True,
            save_hybrid=True
        )
        
        print("평가 결과:")
        if hasattr(metrics, 'box'):
            print(f"  mAP50: {metrics.box.map50:.4f}")
            print(f"  mAP50-95: {metrics.box.map:.4f}")
            print(f"  Precision: {metrics.box.mp:.4f}")
            print(f"  Recall: {metrics.box.mr:.4f}")
        
        if hasattr(metrics, 'seg'):
            print(f"  Seg mAP50: {metrics.seg.map50:.4f}")
            print(f"  Seg mAP50-95: {metrics.seg.map:.4f}")
        
        # wandb에 결과 로그
        if wandb.run is not None:
            wandb.log({
                "eval/mAP50": metrics.box.map50 if hasattr(metrics, 'box') else 0,
                "eval/mAP50-95": metrics.box.map if hasattr(metrics, 'box') else 0,
                "eval/precision": metrics.box.mp if hasattr(metrics, 'box') else 0,
                "eval/recall": metrics.box.mr if hasattr(metrics, 'box') else 0,
                "eval/seg_mAP50": metrics.seg.map50 if hasattr(metrics, 'seg') else 0,
                "eval/seg_mAP50-95": metrics.seg.map if hasattr(metrics, 'seg') else 0,
            })
        
        return metrics
        
    except Exception as e:
        print(f"평가 중 오류 발생: {str(e)}")
        return None

def inference_example(model_path, image_path):
    """
    학습된 모델로 추론 예제
    
    Args:
        model_path (str): 학습된 모델 경로
        image_path (str): 추론할 이미지 경로
    """
    
    print(f"\n=== 추론 예제 ===")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    try:
        model = YOLO(model_path)
        
        # 추론 실행 (기본값 사용)
        results = model(image_path, device=device, save=True, save_txt=True, save_conf=True)
        
        # 결과 출력
        for r in results:
            boxes_count = len(r.boxes) if r.boxes is not None else 0
            masks_count = len(r.masks) if r.masks is not None else 0
            print(f"감지된 객체: {boxes_count}개, 분할된 객체: {masks_count}개")
        
        return results
        
    except Exception as e:
        print(f"추론 중 오류 발생: {str(e)}")
        return None
        '''

In [ ]:
def main():
    """
    메인 실행 함수
    """
    
    print("=== YOLO8 Segmentation 도로 객체 분할 모델 학습기 ===")
    print()
    
    # 1. 경로 입력
    source_images_dir = 'C:/Users/User/Downloads/dataset/images'
    source_labels_dir = 'C:/Users/User/Downloads/dataset/labels'
    changed_labels_dir = 'C:/Users/User/Downloads/dataset/change_labels'
    output_dir = 'C:/Users/User/Downloads/dataset/output'

    convert_dataset_json_to_yolo(source_images_dir, source_labels_dir, changed_labels_dir)

    
    # 경로 검증
    if not all(os.path.exists(p) for p in [source_images_dir, changed_labels_dir]):
        print("오류: 입력 경로가 존재하지 않습니다.")
        return
    
    # 2. wandb 설정
    use_wandb_input = input("wandb 사용하시겠습니까? (y/n) [기본값: y]: ").strip().lower()
    use_wandb = use_wandb_input != 'n'
    
    wandb_initialized = False
    if use_wandb:
        project_name = input("wandb 프로젝트 이름 [기본값: yolo8-road-segmentation]: ").strip()
        if not project_name:
            project_name = "yolo8-road-segmentation"
        
        run_name = input("실행 이름 (선택사항): ").strip()
        run_name = run_name if run_name else None
        
        wandb_initialized = setup_wandb(project_name, run_name)
    
    # 3. 학습 환경 설정
    setup_training_environment()
    
    # 4. 데이터셋 분할
    train_ratio = float(input("훈련 데이터 비율 [기본값: 0.7]: ").strip() or "0.7")
    val_ratio = float(input("검증 데이터 비율 [기본값: 0.2]: ").strip() or "0.2")
    test_ratio = 1.0 - train_ratio - val_ratio
    
    print(f"테스트 데이터 비율: {test_ratio:.2f} (자동 계산)")
    
    train_count, val_count, test_count = split_dataset(
        source_images_dir, changed_labels_dir, output_dir, 
        train_ratio, val_ratio, test_ratio
    )
    
    # 5. 라벨 검증
    train_labels_dir = os.path.join(output_dir, 'labels', 'train')
    if not validate_polygon_labels(train_labels_dir):
        print("라벨 검증에 실패했습니다.")
        return
    
    # 6. YAML 파일 생성
    yaml_path = os.path.join(output_dir, 'dataset.yaml')
    create_dataset_yaml(output_dir, yaml_path)
    
    # 7. 학습 설정
    print(f"\n=== 학습 설정 ===")
    model_size = input("모델 크기 (n/s/m/l/x) [기본값: n]: ").strip().lower()
    if model_size not in ['n', 's', 'm', 'l', 'x']:
        model_size = 'n'
    
    model_name = f'yolov8{model_size}-seg.pt'
    
    epochs = input("에포크 수 [기본값: 100]: ").strip()
    epochs = int(epochs) if epochs.isdigit() else 100
    
    batch_size = input("배치 크기 [기본값: 16]: ").strip()
    batch_size = int(batch_size) if batch_size.isdigit() else 16
    
    # wandb에 설정 로그
    if wandb_initialized:
        wandb.config.update({
            "epochs": epochs,
            "batch_size": batch_size,
            "model_size": model_size,
            "train_count": train_count,
            "val_count": val_count,
            "test_count": test_count,
            "train_ratio": train_ratio,
            "val_ratio": val_ratio,
            "test_ratio": test_ratio
        })
    
    # 8. 모델 학습
    print(f"\n학습을 시작합니다...")
    results, best_model_path = train_yolo_segmentation(
        dataset_yaml_path=yaml_path,
        model_name=model_name,
        epochs=epochs,
        batch_size=batch_size,
        use_wandb=wandb_initialized
    )
    
    if results is None:
        print("학습에 실패했습니다.")
        if wandb_initialized:
            wandb.finish()
        return
    
    # 9. 모델 평가
    print(f"\n모델 평가를 시작합니다...")
    metrics = evaluate_model(best_model_path, yaml_path)
    
    # 10. 추론 예제 (선택사항)
    test_image = input("\n테스트할 이미지 경로 (선택사항): ").strip()
    if test_image and os.path.exists(test_image):
        inference_example(best_model_path, test_image)
    
    # 11. wandb 정리
    if wandb_initialized:
        # 최종 모델 아티팩트로 저장
        artifact = wandb.Artifact('model', type='model')
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
        
        wandb.finish()
        print("✓ wandb 로깅 완료")
    
    print(f"\n=== 학습 완료 ===")
    print(f"최고 성능 모델: {best_model_path}")
    print(f"결과 확인: {results.save_dir if results else 'N/A'}")
    if wandb_initialized:
        print(f"wandb 대시보드에서 상세 결과를 확인하세요.")